# DURAKLATILDI — fold 1 öğretmen eğitimi
25 Ekim 2026 tarihli yarışma kılavuzuna göre teslim CPU'da çalışacak, puan ağırlıkları %60 doğruluk / %25 hız / %15 boyut ve önerilen model boyutu eşiği 35 MB. Bu öğretmen modeli FP16'da bile yaklaşık 56 MB. Bu nedenle **teacher_f1_colab eğitimi yeni seçim kararı verilene kadar başlatılmamalıdır**. Aşağıdaki eğitim hücresi yanlışlıkla çalıştırılmasın diye kilitlendi.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from zipfile import ZipFile
import hashlib
bundle = Path('/content/drive/MyDrive/flower_colab_fold1_bundle.zip')
expected = '48487d4fe298012a71cb9db9fef9a88adb2d1ca641ca378edda045b5d7a8bc44'
assert bundle.is_file(), f'Drive içinde bulunamadı: {bundle}'
digest = hashlib.sha256()
with bundle.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b''):
        digest.update(chunk)
assert digest.hexdigest() == expected, 'Paket bütünlük kontrolü başarısız'
workspace = Path('/content/flower_fold1')
if not (workspace / 'project/train.py').is_file():
    workspace.mkdir(parents=True, exist_ok=True)
    with ZipFile(bundle) as archive:
        archive.extractall(workspace)
project = workspace / 'project'
data = workspace / 'flower_data'
assert (project / 'artifacts/data/folds.jsonl').is_file()
print('Veri ve kod hazır:', project)


In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), 'Colab çalışma zamanı: GPU seçin'
print('GPU:', torch.cuda.get_device_name(0), 'PyTorch:', torch.__version__)
subprocess.run(['python', '-m', 'pip', 'install', '-q', 'timm==1.0.25'], check=True)


In [ ]:
raise RuntimeError('Bu öğretmen fold1 Colab koşusu yeni kılavuza göre duraklatıldı; yeni notebook bekleyin.')
import os, subprocess
run = Path('/content/drive/MyDrive/flower_colab_runs/teacher_f1_colab')
run.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env['HF_HOME'] = str(Path('/content/drive/MyDrive/flower_colab_hf_cache'))
cmd = ['python', 'train.py', '--data', str(data), '--folds', str(project / 'artifacts/data/folds.jsonl'), '--manifest', str(project / 'artifacts/data/manifest.jsonl'), '--run', str(run), '--model', 'teacher', '--fold', '1', '--batch', '2', '--effective-batch', '32', '--head-epochs', '3', '--epochs', '15', '--backbone-lr', '0.0001', '--head-lr', '0.001', '--seed', '42', '--workers', '2']
print('Koşu:', run)
subprocess.run(cmd, cwd=project, env=env, check=True)


In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile
result = Path('/content/drive/MyDrive/teacher_f1_colab_result.zip')
with ZipFile(result, 'w', ZIP_DEFLATED) as archive:
    for name in ('best.pt', 'best_metrics.json', 'config.json', 'history.jsonl'):
        source = run / name
        assert source.is_file(), source
        archive.write(source, name)
print('Tamamlandı. Bu dosyayı yerel projeye geri aktarın:', result)
